<!DOCTYPE html>
<html>
<head>
<style>
.story-box {
    background-color: #f5f5f5;      
    border: 2px solid #333333;       
    padding: 20px;                   
    border-radius: 8px;              
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 16px;
    line-height: 1.6;
    color: #111;
    max-width: 900px;
    margin: 20px auto;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.05);
}
</style>
</head>
<body>

<div class="story-box">
    <strong>📖 Agent-Based Interactive Story Game with LangGraph and LLMs</strong><br><br>
    This project implements an agent-based interactive storytelling game engine that manages game flow using LangGraph, stateful graphs, and Pydantic structured outputs. The core of the system is an intent classification node that leverages a language model to map free-form user input to valid story actions, without requiring menu-based selection or keyword matching. The architecture consists of four specialized nodes (scene rendering, input interpretation, scene transition, and ending detection) connected through conditional edges within a stateful game loop. This approach provides a generalizable pattern for building agent-based systems in domains such as conversational chatbots, automated workflows, and interactive storytelling. The implementation is completely story-agnostic and can adapt to any branching narrative simply by modifying the scene dictionary.
</div>

</body>
</html>

# **1. INSTALL REQUIRED LIBRARIES**

In [ ]:
!pip install langgraph langchain-huggingface transformers torch accelerate -q

# **2. IMPORTS**

In [ ]:
import torch
from typing import TypedDict, Dict, List, Optional
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries installed successfully!")

# **3. LOAD FREE MODEL FROM KAGGLE/HUGGINGFACE**

In [ ]:
print("\n📥 Loading free model (this may take 2-3 minutes)...")

# Using Microsoft's Phi-2 (small, fast, free)
# Alternative: "microsoft/phi-2", "google/gemma-2b", "mistralai/Mistral-7B-v0.1"

MODEL_NAME = "microsoft/phi-2"  # 2.7B parameters - fast for Kaggle

try:
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, 
        trust_remote_code=True,
        clean_up_tokenization_spaces=True
    )
    
    # Add padding token if missing
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,  # Uses less memory
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    
    # Create text generation pipeline
    text_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1
    )
    
    # Wrap for LangChain
    llm = HuggingFacePipeline(pipeline=text_pipeline)
    
    print(f"✅ Successfully loaded {MODEL_NAME}")
    print(f"🎮 AI Story Engine Ready - COMPLETELY FREE!")
    
except Exception as e:
    print(f"⚠️ Error loading {MODEL_NAME}: {e}")
    print("\n📌 Alternative: Using rule-based fallback mode")
    llm = None

# **4. GAME STATE DEFINITION**

In [ ]:
class StoryState(TypedDict):
    """Tracks current game status"""
    current_chapter: str
    player_command: str
    interpreted_action: str
    is_complete: bool
    action_history: List[str]

# **5. STORY DATABASE** 

In [ ]:
STORY_WORLD = {
    # === STARTING AREA ===
    "ancient_library": {
        "title": "📚 The Forgotten Library",
        "description": """You stand in an ancient library hidden beneath a mountain.
Dusty tomes line the walls, glowing with mysterious energy.
Three paths reveal themselves:

📖 NORTH: A spiral staircase leading upward
🗝️ EAST: A door with strange runes
🌊 WEST: A dark tunnel with water sounds

The air hums with magical energy...""",
        "mood": "Mysterious | Adventure",
        "available_actions": {
            "climb_stairs": {
                "text": "Climb the spiral staircase to the upper levels",
                "destination": "astronomy_tower",
                "hint": "upward journey"
            },
            "read_runes": {
                "text": "Study the glowing runes on the eastern door",
                "destination": "enchantment_chamber",
                "hint": "magical discovery"
            },
            "follow_water": {
                "text": "Follow the sound of water through the western tunnel",
                "destination": "underground_lake",
                "hint": "hidden cave"
            }
        }
    },
    
    # === BRANCH 1: TOWER PATH ===
    "astronomy_tower": {
        "title": "🔭 Ancient Observatory",
        "description": """The staircase opens to a breathtaking observatory.
A massive telescope points toward a ceiling that shows moving stars.
Star charts on the wall depict constellations never seen before.

A pedestal holds three crystal orbs:
🟢 Green Orb | 🔵 Blue Orb | 🔴 Red Orb""",
        "mood": "Wonder | Discovery",
        "available_actions": {
            "green_orb": {
                "text": "Touch the green orb - it pulses with life energy",
                "destination": "nature_blessing",
                "hint": "healing magic"
            },
            "blue_orb": {
                "text": "Reach for the blue orb - flowing like water",
                "destination": "wisdom_realm",
                "hint": "knowledge path"
            },
            "red_orb": {
                "text": "Take the red orb - burning with power",
                "destination": "fire_trial",
                "hint": "dangerous path"
            }
        }
    },
    
    # === BRANCH 2: RUNE PATH ===
    "enchantment_chamber": {
        "title": "✨ Chamber of Enchantment",
        "description": """The runic door reveals a room full of floating artifacts.
Swords, rings, and amulets orbit a central crystal.
A ghostly librarian appears: "Choose wisely, seeker."

Three items glow brightest:
⚔️ A silver sword
💍 An emerald ring
📜 A blank scroll""",
        "mood": "Magical | Choice",
        "available_actions": {
            "take_sword": {
                "text": "Grab the floating silver sword",
                "destination": "warrior_path",
                "hint": "combat ready"
            },
            "wear_ring": {
                "text": "Slip on the emerald ring",
                "destination": "shadow_path",
                "hint": "stealth mode"
            },
            "write_scroll": {
                "text": "Take the blank scroll",
                "destination": "scribe_path",
                "hint": "knowledge power"
            }
        }
    },
    
    # === BRANCH 3: WATER PATH ===
    "underground_lake": {
        "title": "💎 Crystal Lake",
        "description": """The tunnel opens to a massive underground lake.
The water glows with bioluminescent algae.
Ancient ruins rise from the water like sleeping giants.

A boat waits at the shore. Three islands visible:
🏝️ Northern island (glowing tree)
🏝️ Eastern island (stone circle)
🏝️ Western island (crystal formations)""",
        "mood": "Serene | Mysterious",
        "available_actions": {
            "go_north": {
                "text": "Row toward the glowing tree island",
                "destination": "tree_of_life",
                "hint": "nature's heart"
            },
            "go_east": {
                "text": "Navigate to the stone circle island",
                "destination": "druid_temple",
                "hint": "ancient rituals"
            },
            "go_west": {
                "text": "Explore the crystal formations island",
                "destination": "crystal_cavern",
                "hint": "mineral wealth"
            }
        }
    },
    
    # === ENDING SCENES (No actions = Game Complete) ===
    "nature_blessing": {
        "title": "🌿 Blessed by Nature",
        "description": """The green orb transports you to an eternal forest.
Dryads welcome you as their protector.
You live in harmony with nature for a thousand years.

[ENDING: Guardian of the Wild]""",
        "mood": "Peaceful Ending",
        "available_actions": {}
    },
    
    "wisdom_realm": {
        "title": "📖 Archive of Ages",
        "description": """The blue orb opens a library of all knowledge.
You spend eternity learning the universe's secrets.
Scholars from across dimensions visit your wisdom.

[ENDING: The Sage Eternal]""",
        "mood": "Enlightened Ending",
        "available_actions": {}
    },
    
    "fire_trial": {
        "title": "🔥 Reborn in Fire",
        "description": """The red orb tests your courage.
You survive the trial and emerge as a phoenix.
Legends speak of your transformation for generations.

[ENDING: The Phoenix Ascendant]""",
        "mood": "Heroic Ending",
        "available_actions": {}
    },
    
    "warrior_path": {
        "title": "⚔️ Sword of Destiny",
        "description": """The silver sword chooses you as its wielder.
You become the greatest warrior of the age.
Evil trembles at your name.

[ENDING: Legendary Warrior]""",
        "mood": "Epic Ending",
        "available_actions": {}
    },
    
    "shadow_path": {
        "title": "🌙 Master of Shadows",
        "description": """The ring grants you invisibility.
You become the unseen protector of the realm.
Mysteries solve themselves when you're near.

[ENDING: Shadow Guardian]""",
        "mood": "Stealth Ending",
        "available_actions": {}
    },
    
    "scribe_path": {
        "title": "✍️ Keeper of Records",
        "description": """The scroll writes your adventures into legend.
Your stories inspire heroes for centuries.
You become the most famous bard in history.

[ENDING: The Storyteller]""",
        "mood": "Creative Ending",
        "available_actions": {}
    },
    
    "tree_of_life": {
        "title": "🌳 Heart of the Forest",
        "description": """The glowing tree shares its ancient wisdom.
You become one with nature's consciousness.
All creatures become your friends.

[ENDING: Forest's Voice]""",
        "mood": "Harmonious Ending",
        "available_actions": {}
    },
    
    "druid_temple": {
        "title": "🕯️ Druid's Secret",
        "description": """Ancient druids initiate you into their circle.
You gain power over seasons and weather.
Balance returns to the land.

[ENDING: Archdruid]""",
        "mood": "Mystical Ending",
        "available_actions": {}
    },
    
    "crystal_cavern": {
        "title": "💠 Crystal Heart",
        "description": """You discover the heart of the mountain.
Crystals sing to you in harmonic resonance.
You become the guardian of earth's treasures.

[ENDING: Gem Lord]""",
        "mood": "Rich Ending",
        "available_actions": {}
    }
}

# **6. FALLBACK INTENT PARSER**

In [ ]:
def simple_intent_match(user_input: str, available_actions: Dict) -> tuple:
    """
    Simple keyword-based matching when LLM is unavailable
    Returns: (matched_key, confidence)
    """
    user_lower = user_input.lower()
    
    # Common action mappings
    keyword_map = {
        "climb_stairs": ["climb", "stairs", "up", "staircase", "upper", "north"],
        "read_runes": ["rune", "door", "east", "study", "symbol", "glyph"],
        "follow_water": ["water", "west", "tunnel", "sound", "lake", "stream"],
        "green_orb": ["green", "orb", "life", "heal", "nature", "green orb"],
        "blue_orb": ["blue", "wisdom", "knowledge", "water", "flow", "blue orb"],
        "red_orb": ["red", "fire", "power", "burn", "flame", "red orb"],
        "take_sword": ["sword", "weapon", "fight", "silver", "blade"],
        "wear_ring": ["ring", "invisible", "stealth", "emerald", "sneak"],
        "write_scroll": ["scroll", "write", "book", "scribe", "paper"],
        "go_north": ["north", "tree", "nature", "island", "northern"],
        "go_east": ["east", "stone", "circle", "druid", "ritual"],
        "go_west": ["west", "crystal", "gem", "mine", "western"]
    }
    
    best_match = None
    best_score = 0
    
    for action_key, action_data in available_actions.items():
        keywords = keyword_map.get(action_key, [])
        # Add action text words
        action_words = action_data['text'].lower().split()
        keywords.extend(action_words)
        
        # Calculate match score
        score = sum(1 for kw in keywords if kw in user_lower)
        
        if score > best_score and score > 0:
            best_score = score
            best_match = action_key
    
    if best_match and best_score >= 1:
        return best_match, best_score / max(1, len(keywords))
    
    return None, 0.0

# **7. SMART INTENT PARSER (WITH FREE LLM)**

In [ ]:
def ai_intent_match(user_input: str, available_actions: Dict, scene_title: str) -> tuple:
    """Use loaded LLM to understand user intent"""
    if llm is None:
        return simple_intent_match(user_input, available_actions)
    
    try:
        # Prepare prompt
        actions_text = "\n".join([
            f"- {key}: {data['text']}" 
            for key, data in available_actions.items()
        ])
        
        prompt = f"""You are a game intent classifier. Current scene: {scene_title}

Player action: "{user_input}"

Available actions:
{actions_text}

Respond with ONLY the action key (e.g., "climb_stairs") that best matches the player's intent.
If no match exists, respond with "NONE".

Response:"""
        
        # Generate response
        response = llm.invoke(prompt)
        
        # Parse response
        result = str(response).strip()
        
        if result in available_actions:
            return result, 0.85
        elif "NONE" in result:
            return None, 0.0
        else:
            # Try to find partial match
            for key in available_actions.keys():
                if key.lower() in result.lower():
                    return key, 0.7
        
        return simple_intent_match(user_input, available_actions)
        
    except Exception as e:
        print(f"⚠️ AI Error: {e}, using fallback")
        return simple_intent_match(user_input, available_actions)

# **8. GAME NODES**

In [ ]:
def display_scene(state: StoryState) -> StoryState:
    """Render current scene"""
    scene = STORY_WORLD[state["current_chapter"]]
    
    print("\n" + "█"*70)
    print(f"📖 {scene['title']}")
    print("█"*70)
    print(f"\n{scene['description']}")
    
    if "mood" in scene:
        print(f"\n🎨 [{scene['mood']}]")
    
    return state

def get_player_intent(state: StoryState) -> StoryState:
    """Get and parse user input"""
    scene = STORY_WORLD[state["current_chapter"]]
    actions = scene.get("available_actions", {})
    
    # Check if game ended
    if not actions:
        state["is_complete"] = True
        return state
    
    # Show available actions (optional hint)
    print("\n" + "─"*50)
    print("💡 Possible actions (be creative!):")
    for key, action in actions.items():
        print(f"   • {action['text']}")
    
    # Get user input
    print("\n" + "▸"*30)
    state["player_command"] = input("\n✨ What do you do? > ").strip()
    
    # Exit condition
    if state["player_command"].lower() in ["quit", "exit", "end", "goodbye"]:
        print("\n👋 Thanks for playing! Safe travels, adventurer!")
        state["is_complete"] = True
        return state
    
    # Parse intent
    matched_key, confidence = ai_intent_match(
        state["player_command"], 
        actions, 
        scene['title']
    )
    
    if matched_key and matched_key in actions:
        state["interpreted_action"] = matched_key
        print(f"\n🎯 I understand: {actions[matched_key]['text']}")
        if confidence > 0:
            print(f"   (confidence: {confidence:.0%})")
    else:
        print("\n🤔 Hmm, I'm not sure what you mean...")
        print("Let me suggest the first option as a fallback.")
        # Suggest first action
        first_key = list(actions.keys())[0]
        state["interpreted_action"] = first_key
        print(f"💡 Try: {actions[first_key]['text']}")
    
    return state

def transition_scene(state: StoryState) -> StoryState:
    """Move to next scene"""
    scene = STORY_WORLD[state["current_chapter"]]
    action_key = state["interpreted_action"]
    
    if action_key in scene.get("available_actions", {}):
        destination = scene["available_actions"][action_key]["destination"]
        old_scene = state["current_chapter"]
        state["current_chapter"] = destination
        
        # Add to history
        if "action_history" not in state:
            state["action_history"] = []
        state["action_history"].append(f"{old_scene} → {destination}")
        
        print(f"\n🚀 Journey continues...")
    else:
        print("\n⚠️ That path seems blocked...")
    
    return state

def check_completion(state: StoryState) -> StoryState:
    """Verify if game reached ending"""
    scene = STORY_WORLD[state["current_chapter"]]
    
    if not scene.get("available_actions"):
        state["is_complete"] = True
        print("\n" + "🏆"*35)
        print(f"✨ GAME COMPLETE ✨")
        print(f"🏆 {scene['title']} 🏆")
        print("🏆"*35)
        
        # Show journey summary
        if "action_history" in state and state["action_history"]:
            print("\n📜 Your journey:")
            for i, step in enumerate(state["action_history"], 1):
                print(f"   {i}. {step}")
    
    return state

# **9. BUILD LANGGRAPH WORKFLOW**

In [ ]:
def build_story_engine():
    """Create the game workflow"""
    
    workflow = StateGraph(StoryState)
    
    # Add nodes
    workflow.add_node("render", display_scene)
    workflow.add_node("parse", get_player_intent)
    workflow.add_node("move", transition_scene)
    workflow.add_node("complete", check_completion)
    
    # Direct edges
    workflow.add_edge("parse", "move")
    workflow.add_edge("move", "complete")
    
    # Conditional routing
    def route_after_render(state: StoryState) -> str:
        if state["is_complete"]:
            return "end"
        return "parse"
    
    workflow.add_conditional_edges(
        "render",
        route_after_render,
        {
            "parse": "parse",
            "end": END
        }
    )
    
    # Start and loop
    workflow.add_edge(START, "render")
    workflow.add_edge("complete", "render")
    
    return workflow.compile()

# **10. LAUNCH GAME**

In [ ]:
def start_game():
    """Initialize and run"""
    
    initial_state = {
        "current_chapter": "ancient_library",
        "player_command": "",
        "interpreted_action": "",
        "is_complete": False,
        "action_history": []
    }
    
    # Welcome
    print("\n" + "🎮"*35)
    print("WELCOME TO THE FORGOTTEN LIBRARY")
    print("🎮"*35)
    print("\n📜 HOW TO PLAY:")
    print("• Type ANYTHING you want to do")
    print("• The AI understands natural language")
    print("• Shape your own destiny")
    print("• Type 'quit' to end your journey")
    print("\n🌙 Your adventure begins at dawn...\n")
    
    try:
        engine = build_story_engine()
        engine.invoke(initial_state)
        
        print("\n" + "✨"*35)
        print("Thank you for playing!")
        print("✨"*35)
        
    except KeyboardInterrupt:
        print("\n\n👋 Adventure interrupted. Until next time!")
    except Exception as e:
        print(f"\n⚠️ Error: {e}")
        print("Try restarting the game.")

# **11. RUN THE GAME**

In [ ]:
if __name__ == "__main__":
    start_game()